# KavachAI — LSTM Training (Phase 3: SOAR Multi-Class)

**Trains 5 disruption level prediction models:**
- `0` = Normal | `1` = Tier 1 (Drip Payout) | `2` = Tier 3 (Force Majeure)

**Hazards:**
- `lstm_aqi.pkl` | `lstm_rain.pkl` | `lstm_heat.pkl` | `lstm_storm.pkl` | `lstm_curfew.pkl` 

**Target:** Multi-class AUC > 0.95 per model
**Window:** 72-hour sliding lookback


In [ ]:
# ── Cell 1: Setup & GPU check ────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — change runtime!"}')

# Install pyarrow for parquet support
!pip install pyarrow fastparquet tqdm -q


In [ ]:
# ── Cell 2: CONFIG ──────────────────────────────────────────────────────
START_DATE = '2022-01-01'
END_DATE   = '2025-01-01'
print(f'Training window: {START_DATE} to {END_DATE}')


In [ ]:
# ── Cell 3: Create directory structure ───────────────────────────────────
import os
os.makedirs('ml/data/raw',       exist_ok=True)
os.makedirs('ml/data/processed', exist_ok=True)
os.makedirs('ml/models',         exist_ok=True)
print('Directories created ✅')


In [ ]:
# ── Cell 4: Fetch Weather Data (Open-Meteo) ───────────────────────────────
import requests, time, json
import pandas as pd
import numpy as np
from pathlib import Path

ZONES = {
    'delhi_rohini':          {'lat': 28.7300, 'lon': 77.1150, 'city': 'delhi_ncr'},
    'delhi_connaught':       {'lat': 28.6315, 'lon': 77.2167, 'city': 'delhi_ncr'},
    'mumbai_andheri':        {'lat': 19.1136, 'lon': 72.8697, 'city': 'mumbai'},
    'bengaluru_koramangala': {'lat': 12.9352, 'lon': 77.6245, 'city': 'bengaluru'},
    'hyderabad_hitech':      {'lat': 17.4435, 'lon': 78.3772, 'city': 'hyderabad'},
    'pune_kothrud':          {'lat': 18.5074, 'lon': 73.8077, 'city': 'pune'},
}

def fetch_weather(zone_id, lat, lon):
    url = (f'https://archive-api.open-meteo.com/v1/archive'
           f'?latitude={lat}&longitude={lon}'
           f'&start_date={START_DATE}&end_date={END_DATE}'
           f'&daily=precipitation_sum,temperature_2m_max,windspeed_10m_max'
           f'&timezone=Asia%2FKolkata')
    r = requests.get(url, timeout=30)
    d = r.json()['daily']
    df = pd.DataFrame({
        'date': pd.to_datetime(d['time']),
        'rainfall_mm': d['precipitation_sum'],
        'temp_max_c':  d['temperature_2m_max'],
        'wind_kmh':    d['windspeed_10m_max'],
        'zone_id':     zone_id,
    })
    return df

weather_dfs = {}
for zone_id, info in ZONES.items():
    df = fetch_weather(zone_id, info['lat'], info['lon'])
    df.to_parquet(f'ml/data/raw/{zone_id}_weather.parquet', index=False)
    weather_dfs[zone_id] = df
    print(f'✅ Weather {zone_id}: {len(df)} days')
    time.sleep(0.3)

print('\nAll weather data fetched ✅')


In [ ]:
# ── Cell 6: Merge + Feature Engineering (3-Class Labels) ─────────────────
def add_features(df):
    df = df.sort_values('date').copy()
    for col in ['max_aqi', 'rainfall_mm', 'max_temp_celsius']:
        df[f'{col}_7d_avg']  = df[col].rolling(7,  min_periods=1).mean()
        df[f'{col}_7d_max']  = df[col].rolling(7,  min_periods=1).max()
        df[f'{col}_30d_avg'] = df[col].rolling(30, min_periods=1).mean()
        df[f'{col}_30d_std'] = df[col].rolling(30, min_periods=1).std().fillna(0)
    df['month_sin'] = np.sin(2 * np.pi * df['date'].dt.month / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['date'].dt.month / 12)
    df['doy_norm']  = df['date'].dt.dayofyear / 365.0
    return df

combined = pd.read_parquet('all_zones_combined.parquet')
all_processed = []
for zone_id, zdf in combined.groupby('zone_code'):
    zdf = add_features(zdf)
    all_processed.append(zdf)
    print(f'✅ {zone_id} processed')

combined = pd.concat(all_processed)
print(f'\nCombined Dataset: {len(combined):,} rows')


In [ ]:
# ── Cell 7: Train Multi-Class LSTMs (SOAR Protocol) ───────────────────────
import pickle, warnings
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
from sklearn.model_selection import train_test_split
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Architecture Parameters
SEQ_LEN    = 3      # 72-hour lookback (daily data)
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 2e-3
HIDDEN     = 128
LAYERS     = 2
DROP       = 0.3
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FEATURE_COLS = [
    'max_aqi','max_aqi_7d_avg','max_aqi_7d_max','max_aqi_30d_avg','max_aqi_30d_std',
    'rainfall_mm','rainfall_mm_7d_avg','rainfall_mm_7d_max','rainfall_mm_30d_avg','rainfall_mm_30d_std',
    'max_temp_celsius','max_temp_celsius_7d_avg','max_temp_celsius_7d_max','max_temp_celsius_30d_avg','max_temp_celsius_30d_std',
    'wind_speed_kmh','month_sin','month_cos','doy_norm',
]

class MultiClassDS(Dataset):
    def __init__(self, X, y): self.X=torch.FloatTensor(X); self.y=torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class KavachLSTM(nn.Module):
    def __init__(self, inp, hid, layers, drop):
        super().__init__()
        self.lstm = nn.LSTM(inp, hid, layers, batch_first=True, bidirectional=True, dropout=drop if layers>1 else 0)
        self.attn = nn.Linear(hid*2, 1)
        self.head = nn.Sequential(nn.Linear(hid*2, 64), nn.ReLU(), nn.Dropout(drop), nn.Linear(64, 3)) # 3 Classes
    def forward(self, x):
        out, _ = self.lstm(x)
        w = torch.softmax(self.attn(out), dim=1)
        ctx = (w * out).sum(dim=1)
        return self.head(ctx)

def make_sequences(df, label_col, scaler=None):
    feat = df[FEATURE_COLS].values.astype(np.float32)
    if scaler is None: scaler = StandardScaler().fit(feat)
    feat = scaler.transform(feat)
    lbl = df[label_col].values.astype(int)
    X, y = [], []
    for i in range(SEQ_LEN, len(feat)):
        X.append(feat[i-SEQ_LEN:i]); y.append(lbl[i])
    return np.array(X), np.array(y), scaler

TASKS = {'aqi':'aqi_class', 'rain':'rain_class', 'heat':'heat_class', 'storm':'storm_class', 'curfew':'curfew_class'}
results = {}

for task, lbl_col in TASKS.items():
    print(f'\n{"="*60}\nTraining SOAR Multi-Class LSTM — {task.upper()}\n{"="*60}')
    df_s = combined.sort_values(['zone_code','date'])
    all_X, all_y, sc = [], [], None
    for _, zdf in df_s.groupby('zone_code'):
        X, y, sc = make_sequences(zdf.reset_index(drop=True), lbl_col, sc)
        all_X.append(X); all_y.append(y)
    
    X_all, y_all = np.concatenate(all_X), np.concatenate(all_y)
    print(f'  Dataset: {len(X_all):,} samples | Distribution: {np.bincount(y_all)}')
    
    Xtr, Xv, ytr, yv = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
    tr_dl = DataLoader(MultiClassDS(Xtr, ytr), BATCH_SIZE, shuffle=True)
    vl_dl = DataLoader(MultiClassDS(Xv, yv), BATCH_SIZE)

    model = KavachLSTM(len(FEATURE_COLS), HIDDEN, LAYERS, DROP).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss() # Standard for multi-class
    
    best_auc, best_state = 0, None
    for ep in range(1, EPOCHS+1):
        model.train()
        for Xb, yb in tr_dl:
            opt.zero_grad(); l = crit(model(Xb.to(DEVICE)), yb.to(DEVICE)); l.backward(); opt.step()
        
        model.eval()
        probs, labs = [], []
        with torch.no_grad():
            for Xb, yb in vl_dl: 
                probs.append(torch.softmax(model(Xb.to(DEVICE)), dim=1).cpu().numpy())
                labs.append(yb.numpy())
        probs, labs = np.concatenate(probs), np.concatenate(labs)
        
        # Multi-class OvR AUC
        try: auc = roc_auc_score(labs, probs, multi_class='ovr')
        except: auc = 0.5
        
        if ep % 5 == 0 or ep == 1: print(f'  Epoch {ep:2d} | Valid AUC: {auc:.4f}')
        if auc > best_auc: best_auc = auc; best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
    
    model.load_state_dict(best_state)
    print(f'  🎯 Best Multi-Class AUC: {best_auc:.4f}')
    
    # Save
    with open(f'ml/models/lstm_{task}.pkl', 'wb') as f:
        pickle.dump({'model_state':model.cpu().state_dict(), 
                     'model_config':{'input_size':len(FEATURE_COLS),'hidden_size':HIDDEN,'num_layers':LAYERS,'dropout':DROP},
                     'seq_len':SEQ_LEN, 'feature_cols':FEATURE_COLS, 'task':task, 'val_auc':best_auc}, f)
    with open(f'ml/models/lstm_scaler_{task}.pkl', 'wb') as f: pickle.dump(sc, f)
    results[task] = best_auc

print('\nModel training complete. All models saved to ml/models/')


In [ ]:
# ── Cell 8: Download Model Files (10 total) ───────────────────────────────
from google.colab import files
import os

TASKS = ['aqi', 'rain', 'heat', 'storm', 'curfew']
for task in TASKS:
    for prefix in ['lstm_', 'lstm_scaler_']:
        path = f'ml/models/{prefix}{task}.pkl'
        if os.path.exists(path):
            files.download(path)
            print(f'⬇️ Downloaded: {path}')


In [ ]:
# ── Cell 8: Download model files ──────────────────────────────────────────
# Download all 6 pkl files — put them in ml/models/ in your repo
from google.colab import files
import os

for task in ['aqi', 'rain', 'heat']:
    for prefix in ['lstm_', 'lstm_scaler_']:
        path = f'ml/models/{prefix}{task}.pkl'
        if os.path.exists(path):
            files.download(path)
            print(f'⬇️  Downloaded: {path}')

print('\nPut all 6 .pkl files in your KavachAI/ml/models/ folder')
print('Then update ml_service to load lstm_aqi.pkl, lstm_rain.pkl, lstm_heat.pkl')
print('instead of the single lstm_disruption.pt')
